# Day 6 — Big Data Engineering
> Date: ___
> Theme: Magic Methods + Custom Exception Hierarchy + Operator Overloading
> Videos: #40 #41 #42

## Videos
- [ ] #40 Magic Methods In Python (8min)
- [ ] #41 Custom Exception In Python (7min)
- [ ] #42 Operator Overloading In Python (9min)

## Theory Card — Magic Methods + Custom Exceptions + Operator Overloading

MAGIC METHODS (Dunders):
These are Python's hooks into built-in operations. You never call __str__() directly.
Python calls it automatically when you do print(obj) or str(obj).
This is how your custom classes behave like built-in types.

Key dunder methods:
__str__   triggered by print(obj) and str(obj). Human readable output.
__repr__  triggered by REPL and repr(obj). Precise developer output.
__len__   triggered by len(obj)
__eq__    triggered by obj1 == obj2
__add__   triggered by obj1 + obj2
__contains__  triggered by x in obj
__getitem__   triggered by obj[key]

__repr__ rule: if you only define one, define __repr__.
Python falls back to it for both str() and REPL display.

IMPORTANT: If you define __eq__, Python sets __hash__ to None automatically.
Your object becomes unhashable — cannot go into a set or be a dict key.
Fix: define __hash__ explicitly if you need both.

CUSTOM EXCEPTION HIERARCHY:
Build a tree of exceptions so you can catch at the right level.

class PipelineError(Exception): pass
class ExtractionError(PipelineError): pass
class TransformationError(PipelineError): pass
class LoadError(PipelineError): pass

Catching PipelineError catches ALL of them.
Catching ExtractionError catches only extraction failures.

Exception chaining:
raise NewError('message') from original_error
Both errors appear in the traceback. Always chain when re-raising.

OPERATOR OVERLOADING:
Only overload when it genuinely makes sense.
Vector + Vector = add components. Clear and logical.
Record + Record = unclear. Do not overload this.

IN DATA ENGINEERING:
Pandas DataFrame uses __len__, __getitem__, __iter__, __contains__ — you will use these patterns.
Custom pipeline exceptions make debugging distributed jobs possible.
When a Spark job fails you need to know exactly which stage and why.

## Video Notes

In [ ]:
# Magic methods —
# Custom exceptions —
# Operator overloading —


## Fixes from Day 5
Do these two quick fixes before new questions.

### Fix A — HW2 transfer method
Your new_department method used an undefined variable. Fix it with a proper parameter.

In [ ]:
# Fix A — corrected Employee with working transfer method
class Employee:
    def __init__(self, name, salary, department):
        self.__name = name
        self.__salary = salary
        self.__department = department

    def get_name(self): return self.__name
    def get_salary(self): return self.__salary
    def get_department(self): return self.__department

    def give_raise(self, amount):
        if 0 < amount < 50000:
            self.__salary += amount
        else:
            raise ValueError(f"Raise amount {amount} must be between 0 and 50000")

    def transfer(self, new_dept):       # parameter added
        self.__department = new_dept    # uses the parameter

    def display(self):
        print(f"Name: {self.__name} | Salary: {self.__salary} | Dept: {self.__department}")

# Test everything
e = Employee("Chandu", 40000, "Planning")
e.display()
e.give_raise(5000)
e.transfer("Engineering")
e.display()

# Show __salary access fails
try:
    print(e.__salary)
except AttributeError as err:
    print(f"Direct access blocked: {err}")


### Fix B — HW3 show TypeError on DataConnector() and print results

In [ ]:
# Fix B — run both connectors and show TypeError
from abc import ABC, abstractmethod

class DataConnector(ABC):
    @abstractmethod
    def connect(self): pass
    @abstractmethod
    def query(self, sql): pass
    @abstractmethod
    def close(self): pass

class MySQLConnector(DataConnector):
    def connect(self): print("MySQL: connected")
    def query(self, sql): print(f"MySQL: running query — {sql}")
    def close(self): print("MySQL: connection closed")

class HDFSConnector(DataConnector):
    def connect(self): print("HDFS: connected to cluster")
    def query(self, sql): print(f"HDFS: scanning files — {sql}")
    def close(self): print("HDFS: connection closed")

def run_pipeline(connector, sql):
    obj = connector()
    obj.connect()
    obj.query(sql)
    obj.close()

print("--- MySQL ---")
run_pipeline(MySQLConnector, "SELECT * FROM sales")

print("\n--- HDFS ---")
run_pipeline(HDFSConnector, "SCAN /data/logs")

print("\n--- Abstract directly ---")
try:
    DataConnector()
except TypeError as e:
    print(f"TypeError: {e}")


## Practice Questions (PQ)

### PQ1 — __str__ vs __repr__
Create a class DataRecord that wraps a dict of field names and values.
__str__ should return a clean readable line like: DataRecord(name=Chandu, age=25)
__repr__ should return the full precise form like: DataRecord({'name': 'Chandu', 'age': 25})
__len__ should return the number of fields.

Create an object. Print it (uses __str__). Type it in REPL style using repr(). Show len(). Explain the difference in a comment.

In [ ]:
# PQ1


### PQ2 — __eq__ and __hash__ problem
Create a LogEntry class with timestamp and message attributes.
Define __eq__ so two LogEntry objects are equal if they have the same timestamp.
Try adding two LogEntry objects to a set — show what error or warning you get.
Fix it by adding __hash__ = lambda self: hash(self.timestamp)
Now prove the set works and deduplicates correctly.

In [ ]:
# PQ2


### PQ3 — Custom exception hierarchy
Create this hierarchy:
PipelineError (base)
  ExtractionError(PipelineError)
  TransformationError(PipelineError)
  LoadError(PipelineError)

Write a function simulate_pipeline(stage) that:
  if stage == 'extract' raises ExtractionError('source file not found')
  if stage == 'transform' raises TransformationError('column id missing at row 42')
  if stage == 'load' raises LoadError('database connection refused')

Call it three times with different stages. Catch each specifically and print the stage name + message.
Then show that catching PipelineError alone catches all three.

In [ ]:
# PQ3


### PQ4 — __add__ and __contains__
Create a Dataset class that wraps a list of dicts (rows of data).
__len__ returns number of rows.
__contains__ returns True if a dict is in the dataset.
__add__ merges two Dataset objects and returns a new Dataset with combined rows.
__getitem__ returns a row by index like dataset[0].

Create two datasets. Check len. Check if a specific row is in it. Add them together. Access a row by index.

In [ ]:
# PQ4


### PQ5 — Operator overloading with Vector
Create a Vector class with x and y attributes.
__add__ adds two vectors component-wise: Vector(1,2) + Vector(3,4) = Vector(4,6)
__mul__ multiplies by a scalar: Vector(2,3) * 3 = Vector(6,9)
__eq__ compares component-wise.
__str__ prints as Vector(x=4, y=6)

Test all four operations and print results.

In [ ]:
# PQ5


## Homework (HW)
> Complete by 5 PM next day in office

### HW1 — DataPipeline with magic methods
Create a DataPipeline class that holds a list of stage names (strings).
__str__ returns: Pipeline: extract -> transform -> load (joining stage names with arrow)
__repr__ returns: DataPipeline(stages=['extract', 'transform', 'load'])
__len__ returns number of stages.
__add__ merges two pipelines and returns a new DataPipeline with combined stages.
__contains__ checks if a stage name is in the pipeline.

Test all of these. Show print(p), repr(p), len(p), 'extract' in p, p1 + p2.

In [ ]:
# HW1


### HW2 — Matrix with operator overloading
Create a Matrix class that wraps a 2D list.
__add__ adds two matrices element-wise. Raise ValueError if shapes do not match.
__mul__ multiplies all elements by a scalar.
__eq__ checks if all elements are equal.
__repr__ prints the matrix in a clean grid format.

Test: add two 2x2 matrices, multiply by scalar 3, compare two matrices.
Test the ValueError by adding two matrices with different shapes.

In [ ]:
# HW2


### HW3 — Full pipeline exception hierarchy
Extend the hierarchy from PQ3 to add:
  CorruptFileError(ExtractionError) — for damaged source files
  SchemaMismatchError(TransformationError) — for column mismatches
  HDFSWriteError(LoadError) — for HDFS write failures

Write a function process_file(filepath) that:
  tries to open the file
  if file does not exist, catches FileNotFoundError and raises CorruptFileError from it (exception chaining)
  if file opens but content is wrong, raises SchemaMismatchError

Call process_file with a fake path. Show the full chained traceback.
Then catch it at the PipelineError level to show hierarchy works.

In [ ]:
# HW3


### HW4 — LogEntry with sorting and deduplication
Create a LogEntry class with level, message, and timestamp attributes.
__str__ returns: [ERROR] 2026-05-01 14:30:00 — connection refused
__repr__ returns the full dict-like representation.
__lt__ compares by timestamp so sorted(entries) works chronologically.
__eq__ + __hash__ based on timestamp so a set deduplicates by timestamp.

Create 6 log entries with some duplicate timestamps.
Sort them. Deduplicate with set. Print both results.

In [ ]:
# HW4


### HW5 — Mixed (magic methods + exceptions + OOP pillars)
Build a SchemaValidator class.
It holds a dict of column_name: expected_type pairs.
__len__ returns number of columns.
__contains__ checks if a column name exists.
__eq__ compares two schemas.
__str__ prints the schema cleanly.

Add a method validate(data_dict) that:
  for each column in the schema checks that the key exists in data_dict
  raises SchemaMismatchError('column X missing') if any column is absent
  raises SchemaMismatchError('column X wrong type') if the type does not match

Test with a valid dict and an invalid dict.
Reuse SchemaMismatchError from HW3 hierarchy.

In [ ]:
# HW5


## Notes

In [ ]:
# What clicked:
# Still confusing:

time_spent = 0
print(f'Time spent: {time_spent} mins')
